<b><font size="6" color="#E8800A">Feature work and dimensionality reduction before selection</font></b><br>

In [8]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("category_encoders") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "category-encoders"]
    )

# `course_helpers.py` sits beside the notebook, and Python does not search
# that folder on its own.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Standard library: `Counter` tallies how often a column is selected,
# `inspect` reads a selector's signature, `textwrap` folds printed text.
import inspect
import textwrap
import warnings
from collections import Counter

# Arrays, frames and plots.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

# Encoders for the categorical columns, and the two association statistics the
# typed filter scores a source column with.
from category_encoders import CountEncoder, TargetEncoder
from scipy.stats import chi2_contingency, pointbiserialr

# Feature extraction first, then the selectors this week compares.
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_selection import (
    RFE,
    RFECV,
    SelectFromModel,
    SelectKBest,
    SequentialFeatureSelector,
    VarianceThreshold,
)

# The imputer that fills each split, the model every technique is scored
# through, the splits it is scored on, and the score reported for it.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit, train_test_split

# The scalers and encoders refitted inside every training split, and the
# warning the solver raises when it stops at `max_iter`.
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)
from sklearn.exceptions import ConvergenceWarning

# The course's shared cleaning log and plot colours.
from course_helpers import CleaningLog, PLOT_BLUE, PLOT_ORANGE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # For reproducibility
FIGSIZE = (6, 5)

# Convergence, one-hot and pandas deprecation warnings are silenced once here.
# The source of each is known and expected.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [6]:
recipe_log = CleaningLog.load("../logs/train_cleaning_log.json")

print(f"{len(recipe_log)} decisions inherited from Week 3\n")
for decision in recipe_log.steps:
    carried = ", ".join(sorted(decision.carries))
    print(f"  {decision.column:28s} {decision.action}")
    print(f"  {'':28s} carries -> {carried}\n")

10 decisions inherited from Week 3

  purchase_date                converted from text to datetime ('%Y-%m-%d')
                               carries -> applied, dtype, format

  all feature columns          drop rows empty in every feature
                               carries -> applied

  prior_mean_purchase_value    recomputed as prior_total_spend / prior_purchase_count
                               carries -> applied

  prior_mean_gap_days          computed from customer_tenure_days and the purchase count
                               carries -> applied

  spend_365d                   inconsistent values blanked; filled from prior_total_spend or 0 where the window holds all of the history or none of it
                               carries -> applied

  prior_std_gap_days, prior_std_purchase_value 0 for one gap or one purchase
                               carries -> applied

  days_since_previous_purchase, spend_365d invalid values -> NaN ('blank the cell')
                

In [ ]:
train = pd.read_csv("../data/interim/train_cleaned.csv",
                    index_col="ID", parse_dates=["purchase_date"])

#We need to decide if we keep the date or not
X = train.drop(columns=["repeat_purchase_90d", "customer_id", "purchase_date"])
y = train["repeat_purchase_90d"]

splitter = GroupShuffleSplit(n_splits=20, test_size=0.2, random_state=RANDOM_STATE)
splits = list(splitter.split(X, y, groups=train["customer_id"]))

print(f"{X.shape[0]:,} rows x {X.shape[1]} source columns")
print(f"{X.isna().sum().sum():,} gaps in the file,"
      f" filled per split from that split's own training rows")
rates = [y.iloc[test_rows].mean() for _, test_rows in splits]
print(f"win rate {y.mean():.4f} overall, {min(rates):.4f}-{max(rates):.4f} per held-out set")
sizes = [len(test_rows) for _, test_rows in splits]
print(f"{len(splits)} 80/20 splits grouped by customer,"
      f" {min(sizes)}-{max(sizes)} rows held out in each")

6,469 rows x 35 source columns
6,276 gaps in the file, filled per split from that split's own training rows
win rate 0.5597 overall, 0.4265-0.6924 per held-out set
20 80/20 splits grouped by customer, 823-1710 rows held out in each
